# FER2013 — ConvNeXt-Tiny Multi-Region Attention (MRA) & Logit Adjustment

Notebook huấn luyện độc lập cho bài toán nhận diện 7 biểu cảm khuôn mặt (FER2013),
được thiết kế để khắc phục trực tiếp các nút thắt đã phát hiện từ nhánh MobileNetV3 PairCorrection:

1. **Kiến trúc mới (`FERMultiRegionNet`):** Thay MobileNetV3-Small (0,93M, BatchNorm) và nhánh cặp lấy
   đặc trưng nông 24 kênh bằng **ConvNeXt-Tiny** (LayerNorm, kernel depthwise 7×7, Stochastic Depth)
   kết hợp **Multi-Region Spatial Attention (MRA)** trên Stage 3 (384 kênh) + Global Stage 4 (768 kênh)
   và **Diversity Loss** khuyến khích 4 đầu attention tập trung vào các vùng khuôn mặt khác nhau.
2. **Chống overfitting thực chất:** Bộ augmentation chuyên cho khuôn mặt (`RandomAffine`, `ColorJitter`,
   `RandomHorizontalFlip`, `RandomErasing 0.25`) kết hợp **MixUp (`alpha=0.4`) + CutMix (`alpha=1.0`)**,
   **Layer-wise LR Decay (LLRD)**, **Model EMA** và huấn luyện **1 pha Cosine duy nhất** (bỏ pha pilot thừa).
3. **Sửa giao thức đánh giá & chống data leakage:** Lọc trùng ảnh exact MD5 và chia tập
   **StratifiedGroupKFold theo perceptual dHash** trên `train/` để tránh rò rỉ ảnh trùng/gần trùng giữa train và val.
   Theo dõi **unsmoothed Val NLL + ECE**, hỗ trợ **Train-time & Post-hoc Logit Adjustment**, **Temperature Scaling**
   và **Horizontal-Flip TTA**. **`RUN_FINAL_TEST = False` mặc định.**
4. **Tối ưu tốc độ pipeline:** Cache ảnh 1 kênh grayscale `uint8` (~63 MiB thay vì 189 MiB), thực hiện
   `RandomAffine`/`ColorJitter` trên ảnh gốc 48×48 **trước khi resize** (giảm ~65× số pixel xử lý trên CPU)
   và tự chọn `num_workers` an toàn theo hệ điều hành.

**Kaggle:** import notebook, gắn dataset `msambare/fer2013`, bật GPU (1× T4) và Internet để tải pretrained
weights `torchvision` lần đầu (không cần cài `timm` hay thư viện ngoài).


In [ ]:
import copy
import hashlib
import json
import math
import os
import platform
import random
import sys
import tempfile
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from contextlib import closing, nullcontext
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from zipfile import BadZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm import tqdm

WORKING_DIR = Path.cwd()
PROJECT_ROOT = next(
    (
        directory
        for directory in (WORKING_DIR, *WORKING_DIR.parents)
        if (directory / 'pyproject.toml').is_file() and (directory / 'src' / 'fer').is_dir()
    ),
    WORKING_DIR,
)

CLASS_NAMES = ('angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise')
NORMALIZE_MEAN = (0.485, 0.456, 0.406)
NORMALIZE_STD = (0.229, 0.224, 0.225)
print(f'Python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__}')


## 1. Cấu hình & Preset đối chứng

Các preset chia sẻ cùng split, pipeline dữ liệu, lịch huấn luyện 1 pha và bộ đo để so sánh công bằng:

| Preset | Backbone | Ảnh | Multi-Region Attn | Train Logit Adj ($	au$) | Mục đích đối chứng |
|---|---|---:|---|---:|---|
| **`convnext_tiny_224_mra` (mặc định)** | ConvNeXt-Tiny | 224 | Bật (4 heads, Stage 3+4) | 0.0 | Kiến trúc đề xuất chính |
| `convnext_tiny_224_global` | ConvNeXt-Tiny | 224 | Tắt (Global Stage 4) | 0.0 | Ablation đo tác dụng thực của MRA |
| `convnext_tiny_224_mra_la` | ConvNeXt-Tiny | 224 | Bật (4 heads) | 0.4 | Cân bằng lớp khó/hiếm lúc train |
| `convnext_tiny_128_mra` | ConvNeXt-Tiny | 128 | Bật (4 heads) | 0.0 | Đối chứng tốc độ vs độ phân giải |
| `resnet34_128_mra` | ResNet-34 | 128 | Bật (4 heads) | 0.0 | Đối chứng backbone CNN chuẩn |
| `resnet18_112_mra` | ResNet-18 | 112 | Bật (4 heads) | 0.0 | Đối chứng nhẹ/nhanh (~11.5M params) |

- Tiêu chí chọn checkpoint mặc định: `selection_metric = 'composite'` (`0.5 * val_acc + 0.5 * val_macro_f1`)
  để cân bằng cả độ chính xác tổng thể lẫn nhóm lớp yếu (`fear`, `disgust`, `sad`). Có thể đổi sang `'macro_f1'`
  hoặc `'accuracy'` trước khi chạy.
- `deduplicate_train = True` lọc ảnh trùng pixel (exact MD5) và gom nhóm near-duplicate bằng perceptual dHash
  khi chia train/val. Đặt `False` nếu muốn tái lập chính xác split cũ chưa lọc trùng.


In [ ]:
EXPERIMENT = 'convnext_tiny_224_mra'

EXPERIMENT_PRESETS = {
    'convnext_tiny_224_mra': {
        'backbone': 'convnext_tiny',
        'image_size': 224,
        'use_mra': True,
        'logit_adj_tau': 0.0,
        'backbone_lr': 1.5e-4,
        'head_lr': 6e-4,
    },
    'convnext_tiny_224_global': {
        'backbone': 'convnext_tiny',
        'image_size': 224,
        'use_mra': False,
        'logit_adj_tau': 0.0,
        'backbone_lr': 1.5e-4,
        'head_lr': 6e-4,
    },
    'convnext_tiny_224_mra_la': {
        'backbone': 'convnext_tiny',
        'image_size': 224,
        'use_mra': True,
        'logit_adj_tau': 0.4,
        'backbone_lr': 1.5e-4,
        'head_lr': 6e-4,
    },
    'convnext_tiny_128_mra': {
        'backbone': 'convnext_tiny',
        'image_size': 128,
        'use_mra': True,
        'logit_adj_tau': 0.0,
        'backbone_lr': 2e-4,
        'head_lr': 8e-4,
    },
    'resnet34_128_mra': {
        'backbone': 'resnet34',
        'image_size': 128,
        'use_mra': True,
        'logit_adj_tau': 0.0,
        'backbone_lr': 2e-4,
        'head_lr': 8e-4,
    },
    'resnet18_112_mra': {
        'backbone': 'resnet18',
        'image_size': 112,
        'use_mra': True,
        'logit_adj_tau': 0.0,
        'backbone_lr': 3e-4,
        'head_lr': 1e-3,
    },
}

CFG = {
    'experiment': EXPERIMENT,
    'run_name': f'fer2013_{EXPERIMENT}',
    'data_dir': None,
    'output_root': '/kaggle/working' if Path('/kaggle').is_dir() else str(PROJECT_ROOT / 'outputs' / 'convnext_mra'),
    'seed': 42,
    'split_seed': 42,
    'validation_fraction': 0.10,
    'val_fold': 0,
    'deduplicate_train': True,
    'backbone': 'convnext_tiny',
    'image_size': 224,
    'pretrained': True,
    'backbone_weights_path': None,
    'use_mra': True,
    'mra_heads': 4,
    'mra_dim': 192,
    'mra_out_dim': 256,
    'diversity_loss_weight': 0.05,
    'stochastic_depth_prob': 0.20,
    'dropout': 0.30,
    'batch_size': 64,
    'epochs': 35,
    'warmup_epochs': 3,
    'early_stop_patience': 10,
    'selection_metric': 'composite',  # 'composite', 'macro_f1', or 'accuracy'
    'backbone_lr': 1.5e-4,
    'head_lr': 6e-4,
    'llrd': 0.80,
    'weight_decay': 2e-2,
    'min_lr_ratio': 0.02,
    'label_smoothing': 0.05,
    'logit_adj_tau': 0.0,
    'mix_probability': 0.50,
    'mixup_alpha': 0.40,
    'cutmix_alpha': 1.00,
    'cutmix_share': 0.50,
    'erasing_p': 0.25,
    'affine_degrees': 12.0,
    'affine_translate': 0.08,
    'affine_scale': (0.90, 1.10),
    'color_jitter': 0.25,
    'gradient_clip': 1.0,
    'use_ema': True,
    'ema_decay': 0.998,
    'eval_tta_flip': True,
    'clean_train_every': 5,
    'clean_train_max_samples': 2048,
    'device': 'cuda' if Path('/kaggle').is_dir() else 'auto',
    'use_amp': True,
    'cpu_threads': 2,
    'num_workers': 'auto',  # 'auto' -> 2 on Linux/Kaggle, 0 on Windows
    'cache_images': True,
    'cache_workers': 8,
    'cache_max_mib': 160,
    'cache_reuse': True,
    'cache_disk': True,
    'cache_revision': 'fer2013-gray-v1',
    'cache_dir': '/kaggle/working/.fer_gray_cache' if Path('/kaggle').is_dir() else str(PROJECT_ROOT / 'outputs' / '.fer_gray_cache'),
}

if EXPERIMENT not in EXPERIMENT_PRESETS:
    raise ValueError(f'Preset không hợp lệ: {EXPERIMENT}')
CFG.update(EXPERIMENT_PRESETS[EXPERIMENT])
# Đặt CFG.update(...) tùy chỉnh tại đây sau khi preset đã cập nhật.


def resolve_num_workers(cfg):
    nw = cfg['num_workers']
    if nw == 'auto':
        return 0 if platform.system() == 'Windows' else min(2, os.cpu_count() or 1)
    return int(nw)


def validate_config(cfg):
    for key in ('seed', 'split_seed'):
        if type(cfg[key]) is not int or not 0 <= cfg[key] < 2**32:
            raise ValueError(f'{key} phải là số nguyên trong [0, 2**32).')
    for key in ('batch_size', 'epochs', 'cpu_threads', 'cache_workers', 'mra_heads', 'mra_dim',
                'mra_out_dim', 'clean_train_every', 'clean_train_max_samples'):
        if type(cfg[key]) is not int or cfg[key] < 1:
            raise ValueError(f'{key} phải là số nguyên dương.')
    if cfg['backbone'] not in ('convnext_tiny', 'resnet34', 'resnet18'):
        raise ValueError('backbone phải là convnext_tiny, resnet34 hoặc resnet18.')
    if cfg['image_size'] not in (112, 128, 224):
        raise ValueError('image_size phải là 112, 128 hoặc 224.')
    if cfg['mra_dim'] % 8 != 0:
        raise ValueError('mra_dim phải chia hết cho 8 (cho GroupNorm).')
    if cfg['mra_heads'] < 2 and cfg['use_mra']:
        raise ValueError('mra_heads cần >= 2 khi bật MRA.')
    if cfg['selection_metric'] not in ('composite', 'macro_f1', 'accuracy'):
        raise ValueError('selection_metric phải là composite, macro_f1 hoặc accuracy.')
    if cfg['device'] not in ('auto', 'cpu', 'cuda'):
        raise ValueError('device phải là auto, cpu hoặc cuda.')
    if cfg['num_workers'] != 'auto' and (type(cfg['num_workers']) is not int or cfg['num_workers'] < 0):
        raise ValueError('num_workers phải là "auto" hoặc số nguyên >= 0.')
    for key in ('early_stop_patience', 'warmup_epochs', 'val_fold'):
        if type(cfg[key]) is not int or cfg[key] < 0:
            raise ValueError(f'{key} phải là số nguyên >= 0.')
    if cfg['warmup_epochs'] >= cfg['epochs']:
        raise ValueError('warmup_epochs phải nhỏ hơn epochs.')
    if not 0 < cfg['validation_fraction'] < 0.5:
        raise ValueError('validation_fraction phải trong (0, 0.5).')
    n_splits = max(2, int(round(1.0 / cfg['validation_fraction'])))
    if cfg['val_fold'] >= n_splits:
        raise ValueError(f'val_fold phải < {n_splits}.')
    for key in ('deduplicate_train', 'pretrained', 'use_mra', 'use_ema', 'eval_tta_flip',
                'use_amp', 'cache_images', 'cache_reuse', 'cache_disk'):
        if type(cfg[key]) is not bool:
            raise ValueError(f'{key} phải là bool.')
    for key in ('dropout', 'stochastic_depth_prob', 'label_smoothing'):
        if not 0 <= cfg[key] < 1:
            raise ValueError(f'{key} phải trong [0, 1).')
    for key in ('mix_probability', 'cutmix_share', 'erasing_p', 'min_lr_ratio', 'llrd', 'ema_decay'):
        if not 0 <= cfg[key] <= 1:
            raise ValueError(f'{key} phải trong [0, 1].')
    if min(cfg['backbone_lr'], cfg['head_lr'], cfg['gradient_clip'], cfg['cache_max_mib']) <= 0:
        raise ValueError('LR, gradient_clip và cache_max_mib phải dương.')
    if min(cfg['weight_decay'], cfg['mixup_alpha'], cfg['cutmix_alpha'],
           cfg['diversity_loss_weight'], cfg['logit_adj_tau'], cfg['color_jitter']) < 0:
        raise ValueError('Các tham số regularization/loss không được âm.')
    if len(cfg['affine_scale']) != 2 or not 0 < cfg['affine_scale'][0] <= cfg['affine_scale'][1]:
        raise ValueError('affine_scale phải là (min_scale, max_scale) hợp lệ.')


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def resolve_runtime_device(cfg):
    if cfg['device'] == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA chưa khả dụng. Bật GPU trên Kaggle hoặc đặt device=cpu khi chạy local.')
    name = cfg['device']
    if name == 'auto':
        name = 'cuda' if torch.cuda.is_available() else 'cpu'
    return torch.device('cuda:0' if name == 'cuda' else name)


validate_config(CFG)
torch.set_num_threads(CFG['cpu_threads'])
DEVICE = resolve_runtime_device(CFG)
print('Device:', DEVICE, '| Effective num_workers:', resolve_num_workers(CFG))
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(DEVICE), '| Visible GPUs:', torch.cuda.device_count())
print(json.dumps(CFG, indent=2, ensure_ascii=False))


## 2. Dữ liệu, cache 1 kênh grayscale, lọc trùng và chia tập chống leakage

Điểm cải tiến chính so với pipeline cũ:
- **Cache 1 kênh grayscale `uint8`:** Lưu đúng ma trận `(H, W)` thay vì nhân 3 kênh RGB (~63 MiB thay vì 189 MiB cho 28.709 ảnh).
- **Transform đúng thứ tự tốc độ:** Chạy `RandomHorizontalFlip`, `RandomAffine`, `ColorJitter` trên ảnh **gốc 48×48 grayscale** trước, sau đó mới `Resize` → lặp 3 kênh → `ToTensor` → `Normalize` → `RandomErasing`.
- **Chống leakage trùng ảnh:** Khi `deduplicate_train=True`:
  1. Tính `md5` trên pixel gốc để loại bỏ bản sao trùng tuyệt đối trong `train/` (nếu hai ảnh giống hệt nhau nhưng **khác nhãn** thì loại cả nhóm nhiễu nhãn đó và ghi nhận số lượng).
  2. Tính **perceptual dHash 64-bit** để gom các ảnh gần trùng vào cùng một nhóm và chia bằng `StratifiedGroupKFold`, bảo đảm ảnh gần trùng không bị tách đôi sang cả train và validation.
- Không đọc thư mục `test/` trong suốt quá trình tạo dữ liệu và huấn luyện.


In [ ]:
def find_data_root(configured_path=None):
    if configured_path:
        root = Path(configured_path).expanduser().resolve()
        if not (root / 'train').is_dir():
            raise FileNotFoundError(f'Không tìm thấy train/ trong {root}')
        return root
    preferred = [
        Path('/kaggle/input/datasets/msambare/fer2013'),
        Path('/kaggle/input/fer2013'),
        PROJECT_ROOT / 'data' / 'fer2013',
    ]
    for root in preferred:
        if (root / 'train' / 'angry').is_dir():
            return root.resolve()
    input_root = Path('/kaggle/input')
    candidates = set()
    for pattern in ('*/train', '*/*/train', '*/*/*/train'):
        for train_path in input_root.glob(pattern):
            if (train_path / 'angry').is_dir():
                candidates.add(train_path.parent.resolve())
    if len(candidates) == 1:
        return candidates.pop()
    raise FileNotFoundError(
        f'Hãy gắn dataset FER2013 và đặt CFG["data_dir"]. Tìm thấy: {sorted(map(str, candidates))}'
    )


def build_transforms(cfg, training=False):
    operations = []
    if training:
        operations += [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomAffine(
                degrees=cfg['affine_degrees'],
                translate=(cfg['affine_translate'], cfg['affine_translate']),
                scale=cfg['affine_scale'],
                interpolation=transforms.InterpolationMode.BILINEAR,
                fill=0,
            ),
        ]
        if cfg['color_jitter'] > 0:
            operations.append(
                transforms.ColorJitter(brightness=cfg['color_jitter'], contrast=cfg['color_jitter'])
            )
    # Resize sau khi đã augment trên ảnh gốc 48x48 để giảm tải CPU.
    operations += [
        transforms.Resize(
            (cfg['image_size'], cfg['image_size']),
            interpolation=transforms.InterpolationMode.BILINEAR,
            antialias=True,
        ),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
    ]
    if training and cfg['erasing_p'] > 0:
        operations.append(
            transforms.RandomErasing(
                p=cfg['erasing_p'], scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0,
            )
        )
    return transforms.Compose(operations)


def check_classes(dataset):
    if dataset.classes != list(CLASS_NAMES):
        raise ValueError(f'Cần các lớp {CLASS_NAMES}, tìm thấy {dataset.classes}')


GRAY_CACHE_REGISTRY = globals().get('GRAY_CACHE_REGISTRY', {})
GRAY_CACHE_FORMAT = 2


def decode_gray_array(path):
    with Image.open(path) as image:
        return np.asarray(image.convert('L'), dtype=np.uint8).copy()


def iter_decoded_gray(paths, workers):
    if workers == 1:
        for path in paths:
            yield path, decode_gray_array(path)
        return
    iterator = iter(paths)
    pool = ThreadPoolExecutor(max_workers=workers)
    pending = {}
    try:
        for _ in range(min(len(paths), workers * 2)):
            path = next(iterator)
            pending[pool.submit(decode_gray_array, path)] = path
        while pending:
            completed, _ = wait(pending, return_when=FIRST_COMPLETED)
            for future in completed:
                path = pending.pop(future)
                image = future.result()
                next_path = next(iterator, None)
                if next_path is not None:
                    pending[pool.submit(decode_gray_array, next_path)] = next_path
                yield path, image
    finally:
        for future in pending:
            future.cancel()
        pool.shutdown(wait=True, cancel_futures=True)


def gray_cache_signature(paths, cfg):
    payload = {
        'format': GRAY_CACHE_FORMAT,
        'revision': cfg['cache_revision'],
        'budget_bytes': int(cfg['cache_max_mib'] * 1024**2),
        'paths': paths,
    }
    return hashlib.sha256(json.dumps(payload, separators=(',', ':')).encode('utf-8')).hexdigest()


class CachedGrayImageLoader:
    """Cache 1-kênh grayscale uint8 trong RAM và tùy chọn pack .npz không nén."""

    def __init__(self, paths, max_mib, workers=8, cache_path=None, signature=None, reuse=True):
        self.images, self.used_bytes = {}, 0
        self.paths = list(dict.fromkeys(map(str, paths)))
        self.budget = int(max_mib * 1024**2)
        self.signature = signature
        self.cache_path = None if cache_path is None else Path(cache_path)
        started = perf_counter()
        if reuse and self.cache_path is not None and self.cache_path.is_file():
            try:
                self.load_pack()
                print(f'Gray cache loaded from disk: {len(self.images):,} images | {self.used_bytes / 1024**2:.1f} MiB | {perf_counter() - started:.1f}s')
                return
            except (OSError, ValueError, KeyError, EOFError, BadZipFile) as exc:
                self.images, self.used_bytes = {}, 0
                print(f'Cache pack unavailable/invalid ({type(exc).__name__}); rebuilding from source.')
        print(f'Reading {len(self.paths):,} grayscale images with {workers} I/O threads...')
        with closing(iter_decoded_gray(self.paths, workers)) as decoded, tqdm(
            total=len(self.paths), desc='Decode grayscale images', unit='img',
        ) as progress:
            for path, image in decoded:
                if self.used_bytes + image.nbytes > self.budget:
                    break
                image.setflags(write=False)
                self.images[path] = image
                self.used_bytes += image.nbytes
                progress.update(1)
        print(f'Cached {len(self.images):,}/{len(self.paths):,} images | Gray {self.used_bytes / 1024**2:.1f} MiB | {perf_counter() - started:.1f}s')
        if self.cache_path is not None:
            self.save_pack()

    def load_pack(self):
        with np.load(self.cache_path, allow_pickle=False) as pack:
            metadata = json.loads(str(pack['metadata'].item()))
            if metadata != {'format': GRAY_CACHE_FORMAT, 'signature': self.signature}:
                raise ValueError('Cache metadata mismatch.')
            indices, shapes, pixels = pack['indices'], pack['shapes'], pack['pixels']
        if indices.ndim != 1 or shapes.shape != (len(indices), 2) or (shapes <= 0).any():
            raise ValueError('Invalid cache arrays.')
        if len(np.unique(indices)) != len(indices) or ((indices < 0) | (indices >= len(self.paths))).any():
            raise ValueError('Invalid cache indices.')
        sizes = [int(h) * int(w) for h, w in shapes]
        if pixels.dtype != np.uint8 or pixels.ndim != 1 or sum(sizes) != pixels.size or pixels.nbytes > self.budget:
            raise ValueError('Invalid cache pixel array.')
        pixels.setflags(write=False)
        offset = 0
        for index, (height, width), size in zip(indices, shapes, sizes):
            self.images[self.paths[int(index)]] = pixels[offset:offset + size].reshape(int(height), int(width))
            offset += size
        self.used_bytes = pixels.nbytes

    def save_pack(self):
        ordered = [(idx, self.images[p]) for idx, p in enumerate(self.paths) if p in self.images]
        indices = np.asarray([idx for idx, _ in ordered], dtype=np.int64)
        shapes = np.asarray([arr.shape[:2] for _, arr in ordered], dtype=np.int64).reshape(-1, 2)
        pixels = np.concatenate([arr.reshape(-1) for _, arr in ordered]) if ordered else np.empty(0, dtype=np.uint8)
        metadata = json.dumps({'format': GRAY_CACHE_FORMAT, 'signature': self.signature})
        temporary = None
        try:
            self.cache_path.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.NamedTemporaryFile(dir=self.cache_path.parent, prefix='graycache-', suffix='.tmp', delete=False) as handle:
                temporary = Path(handle.name)
                np.savez(handle, metadata=np.asarray(metadata), indices=indices, shapes=shapes, pixels=pixels)
            os.replace(temporary, self.cache_path)
            print('Reusable gray cache pack:', self.cache_path)
        except OSError as exc:
            print(f'Could not persist cache ({type(exc).__name__}); RAM cache remains active.')
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    def get_array(self, path):
        path = str(path)
        arr = self.images.get(path)
        return arr if arr is not None else decode_gray_array(path)

    def __call__(self, path):
        arr = self.get_array(path)
        return Image.fromarray(arr, mode='L')


def attach_gray_cache(cfg, *image_datasets):
    if not cfg['cache_images']:
        return None
    paths = list(dict.fromkeys(str(path) for ds in image_datasets for path, _ in ds.samples))
    signature = gray_cache_signature(paths, cfg)
    cache = GRAY_CACHE_REGISTRY.get(signature) if cfg['cache_reuse'] else None
    if cache is None:
        cache_path = Path(cfg['cache_dir']) / f'gray-{signature}.npz' if cfg['cache_disk'] else None
        cache = CachedGrayImageLoader(
            paths,
            cfg['cache_max_mib'],
            workers=cfg['cache_workers'],
            cache_path=cache_path,
            signature=signature,
            reuse=cfg['cache_reuse'],
        )
        GRAY_CACHE_REGISTRY[signature] = cache
    else:
        print(f'Reusing RAM gray cache: {len(cache.images):,}/{len(paths):,} images.')
    for ds in image_datasets:
        ds.loader = cache
    return cache


def compute_pixel_md5_and_dhash(gray_array):
    """Exact MD5 trên pixel gốc + perceptual 64-bit dHash (nhóm near-duplicate)."""
    md5 = hashlib.md5(gray_array.tobytes()).hexdigest()
    small = np.asarray(Image.fromarray(gray_array, mode='L').resize((9, 8), Image.Resampling.BILINEAR), dtype=np.int16)
    diff = (small[:, 1:] > small[:, :-1]).reshape(-1)
    bits = 0
    for bit in diff:
        bits = (bits << 1) | int(bit)
    return md5, f'{bits:016x}'


def analyze_and_deduplicate_train(dataset, cache, enabled=True):
    """Loại bản sao trùng tuyệt đối trong train và gom nhóm dHash để tránh rò rỉ train/val."""
    n = len(dataset.samples)
    if not enabled:
        return list(range(n)), list(range(n)), {
            'deduplicate_enabled': False,
            'raw_train_pool': n,
            'kept_samples': n,
            'removed_same_label_exact_duplicates': 0,
            'removed_conflicting_label_exact_duplicates': 0,
            'unique_dhash_groups': n,
        }
    md5_to_indices = {}
    dhashes = {}
    for idx, (path, label) in enumerate(dataset.samples):
        arr = cache.get_array(path) if cache is not None else decode_gray_array(path)
        md5, dhash = compute_pixel_md5_and_dhash(arr)
        md5_to_indices.setdefault(md5, []).append((idx, label))
        dhashes[idx] = dhash

    kept_indices = []
    removed_same_label = 0
    removed_conflict = 0
    for md5, items in md5_to_indices.items():
        labels = {label for _, label in items}
        if len(labels) > 1:
            # Cùng 1 ảnh nhưng bị gán nhiều nhãn đối nghịch trong train -> loại bỏ để giảm nhiễu nhãn.
            removed_conflict += len(items)
        else:
            kept_indices.append(items[0][0])
            removed_same_label += len(items) - 1
    kept_indices.sort()
    groups = [dhashes[idx] for idx in kept_indices]
    report = {
        'deduplicate_enabled': True,
        'raw_train_pool': n,
        'kept_samples': len(kept_indices),
        'removed_same_label_exact_duplicates': removed_same_label,
        'removed_conflicting_label_exact_duplicates': removed_conflict,
        'unique_dhash_groups': len(set(groups)),
    }
    return kept_indices, groups, report


def build_data(cfg, root, device):
    root = Path(root).resolve()
    augmented = datasets.ImageFolder(root / 'train', transform=build_transforms(cfg, training=True))
    clean = datasets.ImageFolder(root / 'train', transform=build_transforms(cfg, training=False))
    check_classes(augmented)
    cache = attach_gray_cache(cfg, augmented, clean)

    if (root / 'val').is_dir():
        validation_data = datasets.ImageFolder(root / 'val', transform=build_transforms(cfg, training=False))
        check_classes(validation_data)
        attach_gray_cache(cfg, validation_data)
        kept_train, _, dedup_report = analyze_and_deduplicate_train(augmented, cache, cfg['deduplicate_train'])
        train_indices = sorted(map(int, kept_train))
        val_indices = list(range(len(validation_data)))
        validation_source = 'val/'
    else:
        kept_indices, groups, dedup_report = analyze_and_deduplicate_train(augmented, cache, cfg['deduplicate_train'])
        kept_targets = [augmented.targets[i] for i in kept_indices]
        if cfg['deduplicate_train']:
            n_splits = max(2, int(round(1.0 / cfg['validation_fraction'])))
            sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg['split_seed'])
            folds = list(sgkf.split(kept_indices, kept_targets, groups))
            tr_pos, va_pos = folds[cfg['val_fold']]
            train_indices = sorted(kept_indices[i] for i in tr_pos)
            val_indices = sorted(kept_indices[i] for i in va_pos)
            validation_source = f'StratifiedGroupKFold(dHash, fold {cfg["val_fold"]}/{n_splits})'
        else:
            tr_pos, va_pos = train_test_split(
                np.arange(len(kept_indices)),
                test_size=cfg['validation_fraction'],
                random_state=cfg['split_seed'],
                stratify=kept_targets,
            )
            train_indices = sorted(kept_indices[int(i)] for i in tr_pos)
            val_indices = sorted(kept_indices[int(i)] for i in va_pos)
            validation_source = 'stratified split from train/ (no dedup)'
        validation_data = clean

    expected_labels = set(range(len(CLASS_NAMES)))
    if {augmented.targets[i] for i in train_indices} != expected_labels:
        raise ValueError('Train sau khi chia không đủ 7 lớp.')
    if {validation_data.targets[i] for i in val_indices} != expected_labels:
        raise ValueError('Validation sau khi chia không đủ 7 lớp.')

    train_paths = {augmented.samples[i][0] for i in train_indices}
    val_paths = {validation_data.samples[i][0] for i in val_indices}
    if train_paths & val_paths:
        raise ValueError('Train và validation có đường dẫn trùng nhau.')

    clean_indices = train_indices
    if cfg['clean_train_max_samples'] < len(train_indices):
        _, selected = train_test_split(
            train_indices,
            test_size=cfg['clean_train_max_samples'],
            random_state=cfg['split_seed'],
            stratify=np.asarray(clean.targets)[train_indices],
        )
        clean_indices = sorted(map(int, selected))

    num_workers = resolve_num_workers(cfg)
    loader_kwargs = {
        'batch_size': cfg['batch_size'],
        'num_workers': num_workers,
        'pin_memory': device.type == 'cuda',
        'worker_init_fn': seed_worker,
        'persistent_workers': num_workers > 0,
    }
    generator = torch.Generator().manual_seed(cfg['seed'])
    loaders = {
        'train': DataLoader(Subset(augmented, train_indices), shuffle=True, drop_last=False, generator=generator, **loader_kwargs),
        'val': DataLoader(Subset(validation_data, val_indices), shuffle=False, **loader_kwargs),
        'clean_train': DataLoader(Subset(clean, clean_indices), shuffle=False, **loader_kwargs),
    }

    records = []
    for split, ds, selected in (('train', augmented, train_indices), ('val', validation_data, val_indices)):
        for index in selected:
            path, label = ds.samples[index]
            records.append({
                'split': split,
                'path': str(Path(path).relative_to(root)),
                'label': label,
                'class': CLASS_NAMES[label],
            })
    manifest = pd.DataFrame(records)
    summary = {
        'train_size': len(train_indices),
        'val_size': len(val_indices),
        'clean_train_size': len(clean_indices),
        'validation_source': validation_source,
        'deduplication': dedup_report,
    }
    return loaders, manifest, summary, cache


def manifest_fingerprint(manifest):
    records = manifest[['split', 'path', 'label']].to_dict('records')
    for row in records:
        row['path'] = row['path'].replace('\\', '/')
    payload = json.dumps(records, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


In [ ]:
seed_everything(CFG['seed'])
DATA_ROOT = find_data_root(CFG['data_dir'])
loaders, split_manifest, data_summary, gray_cache = build_data(CFG, DATA_ROOT, DEVICE)
CLASS_COUNTS = np.bincount(
    split_manifest.loc[split_manifest['split'] == 'train', 'label'].to_numpy(), minlength=7,
).tolist()
SPLIT_FINGERPRINT = manifest_fingerprint(split_manifest)
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR = Path(CFG['output_root']) / f'{CFG["run_name"]}_{stamp}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / 'run_config.json').write_text(json.dumps(CFG, indent=2), encoding='utf-8')
split_manifest.to_csv(RUN_DIR / 'split_manifest.csv', index=False)
print('Output:', RUN_DIR)
print('Dataset:', DATA_ROOT)
print(json.dumps(data_summary, indent=2))
print(pd.crosstab(split_manifest['class'], split_manifest['split']).reindex(CLASS_NAMES))
print('Train class counts:', CLASS_COUNTS)
print('Split fingerprint:', SPLIT_FINGERPRINT)


## 3. Kiến trúc `FERMultiRegionNet`: ConvNeXt-Tiny + Multi-Region Spatial Attention (Stage 3 + Stage 4)

```text
Input (B × 3 × H × H)  [H ∈ {112, 128, 224}]
  │
  ▼
Backbone (ConvNeXt-Tiny / ResNet-34 / ResNet-18, pretrained ImageNet)
  ├── Stage 3 feature map (ConvNeXt: 384×14×14 @224 hoặc 384×8×8 @128)
  │      │
  │      ▼  MultiRegionAttentionPool (GroupNorm + DWConv3×3 + K=4 spatial attention maps)
  │      ├── Diversity Loss trên độ tương đồng cosine giữa K=4 bản đồ attention (chống sụp về 1 vùng)
  │      └── Concatenate K=4 vector vùng → LayerNorm → Linear(K*192 → 256) → GELU
  │
  └── Stage 4 feature map (ConvNeXt: 768×7×7 @224 hoặc 768×4×4 @128)
         │
         ▼  Global Average Pool → LayerNorm(768)
  │
  ▼
Fusion [Global (768) ⊕ Multi-Region (256)] → Dropout(0.3) → Linear(1024 → 7)
```

**Vì sao kiến trúc này xử lý các hạn chế của nhánh cũ:**
- **LayerNorm + Stochastic Depth (`0.20`)** của ConvNeXt miễn nhiễm với lệch thống kê BatchNorm khi dùng MixUp/CutMix mạnh và dữ liệu mất cân bằng.
- **Đặc trưng vùng ở tầng ngữ nghĩa sâu (Stage 3, 384 kênh)** thay cho `features[3]` quá nông (24 kênh) ở MobileNetV3-Small.
- **Không cần pha pilot chọn cặp từ validation** và **không có xung đột mục tiêu** giữa residual correction và BCE độc lập.


In [ ]:
class MultiRegionAttentionPool(nn.Module):
    """K đầu Spatial Attention độc lập trên Stage 3 kèm Diversity Loss khuyến khích tách vùng khuôn mặt."""

    def __init__(self, in_channels, region_dim=192, out_dim=256, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.region_dim = region_dim
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, region_dim, kernel_size=1, bias=False),
            nn.GroupNorm(8, region_dim),
            nn.GELU(),
        )
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(region_dim, region_dim, kernel_size=3, padding=1, groups=region_dim, bias=False),
            nn.GroupNorm(8, region_dim),
            nn.GELU(),
            nn.Conv2d(region_dim, num_heads, kernel_size=1),
        )
        self.fuse = nn.Sequential(
            nn.LayerNorm(num_heads * region_dim),
            nn.Linear(num_heads * region_dim, out_dim),
            nn.GELU(),
        )

    def forward(self, feat):
        x = self.proj(feat)                                 # (B, C, H, W)
        attn_logits = self.spatial_attn(x).flatten(2)       # (B, K, N)
        attn = attn_logits.float().softmax(dim=-1)          # (B, K, N)
        region_vecs = torch.einsum('bkn,bcn->bkc', attn, x.flatten(2).float())  # (B, K, C)
        fused = self.fuse(region_vecs.reshape(feat.size(0), -1))

        # Diversity loss: giảm độ tương đồng cosine giữa các cặp đầu attention (i != j).
        normed = F.normalize(attn, p=2, dim=-1, eps=1e-6)
        gram = torch.bmm(normed, normed.transpose(1, 2))    # (B, K, K)
        k = self.num_heads
        off_diag_mask = ~torch.eye(k, dtype=torch.bool, device=gram.device)
        diversity_loss = gram[:, off_diag_mask].mean()
        return fused, diversity_loss, attn


class FERMultiRegionNet(nn.Module):
    def __init__(
        self,
        backbone='convnext_tiny',
        image_size=224,
        use_mra=True,
        mra_heads=4,
        mra_dim=192,
        mra_out_dim=256,
        dropout=0.30,
        stochastic_depth_prob=0.20,
        pretrained=False,
        backbone_weights_path=None,
    ):
        super().__init__()
        self.backbone_name = backbone
        self.image_size = image_size
        self.use_mra = use_mra
        self.mra_heads = mra_heads
        self.mra_dim = mra_dim
        self.mra_out_dim = mra_out_dim
        self.dropout_p = dropout
        self.stochastic_depth_prob = stochastic_depth_prob

        use_tv_weights = pretrained and not backbone_weights_path
        try:
            if backbone == 'convnext_tiny':
                w = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if use_tv_weights else None
                net = models.convnext_tiny(weights=w, stochastic_depth_prob=stochastic_depth_prob)
                if pretrained and backbone_weights_path:
                    net.load_state_dict(torch.load(backbone_weights_path, map_location='cpu', weights_only=True), strict=True)
                self.features = net.features
                self.c3, self.c4 = 384, 768
            elif backbone == 'resnet34':
                w = models.ResNet34_Weights.IMAGENET1K_V1 if use_tv_weights else None
                net = models.resnet34(weights=w)
                if pretrained and backbone_weights_path:
                    net.load_state_dict(torch.load(backbone_weights_path, map_location='cpu', weights_only=True), strict=True)
                self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
                self.layer1, self.layer2, self.layer3, self.layer4 = net.layer1, net.layer2, net.layer3, net.layer4
                self.c3, self.c4 = 256, 512
            elif backbone == 'resnet18':
                w = models.ResNet18_Weights.IMAGENET1K_V1 if use_tv_weights else None
                net = models.resnet18(weights=w)
                if pretrained and backbone_weights_path:
                    net.load_state_dict(torch.load(backbone_weights_path, map_location='cpu', weights_only=True), strict=True)
                self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
                self.layer1, self.layer2, self.layer3, self.layer4 = net.layer1, net.layer2, net.layer3, net.layer4
                self.c3, self.c4 = 256, 512
            else:
                raise ValueError(f'Backbone không hỗ trợ: {backbone}')
        except Exception as exc:
            raise RuntimeError(
                f'Không tải được pretrained weights cho {backbone}. Bật Internet trên Kaggle hoặc đặt backbone_weights_path.'
            ) from exc

        self.global_norm = nn.LayerNorm(self.c4)
        self.mra = MultiRegionAttentionPool(self.c3, mra_dim, mra_out_dim, mra_heads) if use_mra else None
        head_in = self.c4 + (mra_out_dim if use_mra else 0)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(head_in, len(CLASS_NAMES))
        nn.init.trunc_normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def extract_stages(self, x):
        if self.backbone_name == 'convnext_tiny':
            s3 = None
            for idx, block in enumerate(self.features):
                x = block(x)
                if idx == 5:
                    s3 = x
            return s3, x
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        s3 = self.layer3(x)
        s4 = self.layer4(s3)
        return s3, s4

    def forward(self, x):
        if x.ndim != 4 or tuple(x.shape[1:]) != (3, self.image_size, self.image_size):
            raise ValueError(f'Cần tensor B x 3 x {self.image_size} x {self.image_size}.')
        s3, s4 = self.extract_stages(x)
        global_vec = self.global_norm(s4.mean(dim=(2, 3)).float())
        if self.mra is not None:
            mra_vec, div_loss, attn = self.mra(s3)
            feat = torch.cat([global_vec, mra_vec], dim=1)
        else:
            div_loss = global_vec.new_zeros(())
            attn = None
            feat = global_vec
        logits = self.classifier(self.dropout(feat)).float()
        return {'logits': logits, 'diversity_loss': div_loss, 'attention': attn}

    def stage_param_groups(self, backbone_lr, head_lr, llrd, weight_decay):
        """Chia nhóm tham số theo tầng sâu với Layer-wise LR Decay và tắt weight decay cho bias/norm."""
        if self.backbone_name == 'convnext_tiny':
            stage_map = [
                ('features.0.', 0), ('features.1.', 1),
                ('features.2.', 2), ('features.3.', 2),
                ('features.4.', 3), ('features.5.', 3),
                ('features.6.', 4), ('features.7.', 4),
            ]
            max_stage = 4
        else:
            stage_map = [('stem.', 0), ('layer1.', 1), ('layer2.', 2), ('layer3.', 3), ('layer4.', 4)]
            max_stage = 4

        buckets = {}
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            stage = None
            for prefix, idx in stage_map:
                if name.startswith(prefix):
                    stage = idx
                    break
            if stage is None:
                lr = head_lr
                group_name = 'head'
            else:
                lr = backbone_lr * (llrd ** (max_stage - stage))
                group_name = f'stage_{stage}'
            no_decay = param.ndim <= 1 or name.endswith('.bias')
            wd = 0.0 if no_decay else weight_decay
            key = (group_name, lr, wd)
            buckets.setdefault(key, []).append(param)

        return [
            {'params': params, 'lr': lr, 'weight_decay': wd, 'name': group_name}
            for (group_name, lr, wd), params in buckets.items()
        ]


def model_spec(cfg):
    return {
        'model_type': 'fer_multi_region_net',
        'backbone': cfg['backbone'],
        'image_size': cfg['image_size'],
        'use_mra': cfg['use_mra'],
        'mra_heads': cfg['mra_heads'],
        'mra_dim': cfg['mra_dim'],
        'mra_out_dim': cfg['mra_out_dim'],
        'dropout': cfg['dropout'],
        'stochastic_depth_prob': cfg['stochastic_depth_prob'],
    }


def build_model_from_spec(spec, pretrained=False, backbone_weights_path=None):
    if spec.get('model_type') != 'fer_multi_region_net':
        raise ValueError('Sai kiến trúc checkpoint.')
    kwargs = {k: v for k, v in spec.items() if k != 'model_type'}
    return FERMultiRegionNet(**kwargs, pretrained=pretrained, backbone_weights_path=backbone_weights_path)


def build_model(cfg, pretrained=False):
    return build_model_from_spec(
        model_spec(cfg),
        pretrained=pretrained,
        backbone_weights_path=cfg['backbone_weights_path'] if pretrained else None,
    )


def save_checkpoint(model, cfg, epoch, metrics, class_counts, path, extra=None):
    payload = {
        'format_version': 2,
        'state_dict': model.state_dict(),
        'model_config': model_spec(cfg),
        'classes': list(CLASS_NAMES),
        'class_counts': list(map(int, class_counts)),
        'preprocess': {
            'image_size': model.image_size,
            'channels': 3,
            'grayscale': True,
            'mean': list(NORMALIZE_MEAN),
            'std': list(NORMALIZE_STD),
        },
        'epoch': int(epoch),
        'validation': metrics,
        'posthoc': extra or {},
    }
    torch.save(payload, path)


def load_trained_model(checkpoint_path, device):
    payload = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if not isinstance(payload, dict) or payload.get('format_version') != 2:
        raise ValueError('Checkpoint không đúng format_version=2.')
    spec = payload['model_config']
    expected_pre = {
        'image_size': spec['image_size'],
        'channels': 3,
        'grayscale': True,
        'mean': list(NORMALIZE_MEAN),
        'std': list(NORMALIZE_STD),
    }
    if payload.get('classes') != list(CLASS_NAMES) or payload.get('preprocess') != expected_pre:
        raise ValueError('Nhãn hoặc tiền xử lý trong checkpoint không khớp.')
    model = build_model_from_spec(spec, pretrained=False)
    model.load_state_dict(payload['state_dict'], strict=True)
    return model.to(device).eval(), payload


## 4. Loss (Logit Adjustment + Diversity), MixUp/CutMix, EMA & Calibration

- **Logit-Adjusted Cross-Entropy (Menon et al., ICLR 2021):** Khi `logit_adj_tau > 0`, cộng `tau * log(prior)`
  vào logit khi tính loss train để đẩy biên quyết định hỗ trợ các lớp thiểu số (`disgust`) mà không gây bùng nổ gradient.
- **MixUp (`alpha=0.4`) + CutMix (`alpha=1.0`):** Khắc phục lỗi `Beta(0.1, 0.1)` trộn <1% của notebook trước;
  mỗi batch áp dụng xác suất `mix_probability=0.5`, chia đều 50/50 giữa MixUp và CutMix.
- **Model EMA (`decay=0.998`):** Duy trì trung bình trượt trọng số giúp ổn định trên dữ liệu nhiễu nhãn như FER2013.
- **Đánh giá độ hiệu chuẩn:** Đo **unsmoothed NLL** và **ECE (15 bins)** độc lập với `label_smoothing`.


In [ ]:
class ModelEMA:
    def __init__(self, model, decay=0.998):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        self.updates = 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        # Warmup decay nhẹ trong những step đầu để EMA bắt kịp trọng số nhanh.
        d = min(self.decay, (1.0 + self.updates) / (10.0 + self.updates))
        ema_state = self.ema.state_dict()
        model_state = model.state_dict()
        for key, v_ema in ema_state.items():
            v_model = model_state[key]
            if v_ema.dtype.is_floating_point:
                v_ema.mul_(d).add_(v_model.detach(), alpha=1.0 - d)
            else:
                v_ema.copy_(v_model)


class FERLoss(nn.Module):
    def __init__(self, class_counts, label_smoothing=0.05, logit_adj_tau=0.0, diversity_weight=0.05):
        super().__init__()
        counts = torch.as_tensor(class_counts, dtype=torch.float32)
        if counts.shape != (len(CLASS_NAMES),) or (counts <= 0).any():
            raise ValueError('class_counts phải dương cho đủ 7 lớp.')
        priors = counts / counts.sum()
        self.register_buffer('log_priors', priors.log())
        self.label_smoothing = label_smoothing
        self.logit_adj_tau = logit_adj_tau
        self.diversity_weight = diversity_weight

    def _ce(self, logits, targets):
        adj_logits = logits + self.logit_adj_tau * self.log_priors if self.logit_adj_tau > 0 else logits
        return F.cross_entropy(adj_logits, targets, label_smoothing=self.label_smoothing)

    def forward(self, outputs, targets_a, targets_b=None, lam=1.0):
        logits = outputs['logits']
        cls_loss = lam * self._ce(logits, targets_a)
        if targets_b is not None and lam < 1.0:
            cls_loss = cls_loss + (1.0 - lam) * self._ce(logits, targets_b)
        div_loss = outputs['diversity_loss']
        total = cls_loss + self.diversity_weight * div_loss
        return total, cls_loss, div_loss


def apply_mixup_or_cutmix(inputs, targets, cfg):
    if cfg['mix_probability'] <= 0 or np.random.random() >= cfg['mix_probability']:
        return inputs, targets, targets, 1.0
    use_cutmix = cfg['cutmix_alpha'] > 0 and np.random.random() < cfg['cutmix_share']
    index = torch.randperm(inputs.size(0), device=inputs.device)
    if use_cutmix:
        lam = float(np.random.beta(cfg['cutmix_alpha'], cfg['cutmix_alpha']))
        _, _, h, w = inputs.shape
        cut_ratio = math.sqrt(max(0.0, 1.0 - lam))
        cut_w, cut_h = int(w * cut_ratio), int(h * cut_ratio)
        cx, cy = np.random.randint(w), np.random.randint(h)
        x1 = max(0, cx - cut_w // 2)
        y1 = max(0, cy - cut_h // 2)
        x2 = min(w, cx + cut_w // 2)
        y2 = min(h, cy + cut_h // 2)
        mixed = inputs.clone()
        mixed[:, :, y1:y2, x1:x2] = inputs[index, :, y1:y2, x1:x2]
        lam = 1.0 - ((x2 - x1) * (y2 - y1) / float(w * h))
        return mixed, targets, targets[index], lam
    if cfg['mixup_alpha'] <= 0:
        return inputs, targets, targets, 1.0
    lam = float(np.random.beta(cfg['mixup_alpha'], cfg['mixup_alpha']))
    mixed = lam * inputs + (1.0 - lam) * inputs[index]
    return mixed, targets, targets[index], lam


def lr_multiplier(epoch_index, epochs, warmup, minimum):
    # epoch_index: 0..epochs-1; warmup epochs tăng tuyến tính (1..warmup)/warmup rồi giảm cosine mượt.
    if warmup > 0 and epoch_index < warmup:
        return (epoch_index + 1) / warmup
    decay_steps = max(1, epochs - warmup)
    progress = min(1.0, max(0.0, (epoch_index - warmup + 1) / decay_steps))
    return minimum + (1.0 - minimum) * 0.5 * (1.0 + math.cos(math.pi * progress))


def build_optimizer(model, cfg):
    param_groups = model.stage_param_groups(
        backbone_lr=cfg['backbone_lr'],
        head_lr=cfg['head_lr'],
        llrd=cfg['llrd'],
        weight_decay=cfg['weight_decay'],
    )
    optimizer = torch.optim.AdamW(param_groups)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda ep: lr_multiplier(ep, cfg['epochs'], cfg['warmup_epochs'], cfg['min_lr_ratio']),
    )
    return optimizer, scheduler


def amp_context(device, enabled):
    return torch.autocast('cuda', dtype=torch.float16) if enabled and device.type == 'cuda' else nullcontext()


def compute_ece(probs, targets, n_bins=15):
    confidences, predictions = probs.max(dim=1)
    accuracies = predictions.eq(targets).float()
    bins = torch.linspace(0.0, 1.0, n_bins + 1, device=probs.device)
    ece = probs.new_zeros(())
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.any():
            ece = ece + (mask.float().mean()) * (accuracies[mask].mean() - confidences[mask].mean()).abs()
    return float(ece.item())


def compute_metrics_from_logits(logits, targets):
    logits_f = logits.float().cpu()
    targets_l = targets.long().cpu()
    nll = float(F.cross_entropy(logits_f, targets_l).item())
    probs = logits_f.softmax(dim=1)
    ece = compute_ece(probs, targets_l)
    preds = logits_f.argmax(dim=1).numpy()
    true = targets_l.numpy()
    cm = confusion_matrix(true, preds, labels=list(range(len(CLASS_NAMES))))
    supports = cm.sum(1)
    recalls = np.divide(cm.diagonal(), supports, out=np.zeros(len(CLASS_NAMES), dtype=float), where=supports > 0)
    acc = float(np.mean(true == preds))
    macro_f1 = float(f1_score(true, preds, labels=list(range(len(CLASS_NAMES))), average='macro', zero_division=0))
    return {
        'nll': nll,
        'ece': ece,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'balanced_accuracy': float(recalls[supports > 0].mean()),
        'composite': 0.5 * acc + 0.5 * macro_f1,
    }


def fit_temperature(logits, targets):
    """Tối ưu nhiệt độ T > 0 trên validation để giảm NLL/ECE (không làm đổi argmax)."""
    # Clone ngoài inference_mode để chuyển inference tensor về tensor thường cho autograd/LBFGS.
    logits_f = logits.detach().to(device='cpu', dtype=torch.float32).clone()
    targets_l = targets.detach().to(device='cpu', dtype=torch.long).clone()
    log_t = nn.Parameter(torch.zeros(()))
    opt = torch.optim.LBFGS([log_t], lr=0.25, max_iter=50)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits_f / log_t.exp().clamp(0.05, 10.0), targets_l)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().clamp(0.05, 10.0).item())


In [ ]:
@torch.inference_mode()
def collect_logits(model, loader, device, cfg, tta_flip=False):
    model.eval()
    all_logits, all_targets = [], []
    div_sum, count = 0.0, 0
    for inputs, targets in tqdm(loader, desc='Evaluate', leave=False):
        inputs = inputs.to(device, non_blocking=True)
        with amp_context(device, cfg['use_amp']):
            out = model(inputs)
            logits = out['logits'].float()
            if tta_flip:
                logits_flip = model(torch.flip(inputs, dims=[3]))['logits'].float()
                logits = 0.5 * (logits + logits_flip)
        if not torch.isfinite(logits).all().item():
            raise FloatingPointError('Logits không hữu hạn khi evaluate.')
        all_logits.append(logits.cpu())
        all_targets.append(targets.cpu())
        div_sum += float(out['diversity_loss'].item()) * len(targets)
        count += len(targets)
    if count == 0:
        raise ValueError('Tập đánh giá rỗng.')
    logits_cat = torch.cat(all_logits, dim=0)
    targets_cat = torch.cat(all_targets, dim=0)
    metrics = compute_metrics_from_logits(logits_cat, targets_cat)
    metrics['diversity_loss'] = div_sum / count
    return metrics, logits_cat, targets_cat


def selection_key(metrics, metric_name):
    if metric_name == 'composite':
        return (metrics['composite'], metrics['macro_f1'], metrics['accuracy'])
    if metric_name == 'macro_f1':
        return (metrics['macro_f1'], metrics['accuracy'])
    return (metrics['accuracy'], metrics['macro_f1'])


def train_one_epoch(model, ema, loader, optimizer, scaler, criterion, device, cfg, epoch):
    model.train()
    totals = {'loss': 0.0, 'cls_loss': 0.0, 'div_loss': 0.0, 'correct': 0.0, 'count': 0}
    progress = tqdm(loader, desc=f'Epoch {epoch}/{cfg["epochs"]}')
    for inputs, targets in progress:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        mixed_inputs, targets_a, targets_b, lam = apply_mixup_or_cutmix(inputs, targets, cfg)
        optimizer.zero_grad(set_to_none=True)
        with amp_context(device, cfg['use_amp']):
            outputs = model(mixed_inputs)
            loss, cls_loss, div_loss = criterion(outputs, targets_a, targets_b, lam)
        if not torch.isfinite(loss).item():
            raise FloatingPointError(f'Loss không hữu hạn ở epoch {epoch}.')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update(model)

        predicted = outputs['logits'].detach().argmax(dim=1)
        bs = len(targets)
        totals['correct'] += lam * predicted.eq(targets_a).sum().item() + (1.0 - lam) * predicted.eq(targets_b).sum().item()
        totals['count'] += bs
        totals['loss'] += float(loss.item()) * bs
        totals['cls_loss'] += float(cls_loss.item()) * bs
        totals['div_loss'] += float(div_loss.item()) * bs
        progress.set_postfix(loss=f'{loss.item():.4f}')
    n = totals['count']
    return {
        'loss': totals['loss'] / n,
        'cls_loss': totals['cls_loss'] / n,
        'div_loss': totals['div_loss'] / n,
        'accuracy': totals['correct'] / n,
    }


def run_training(model, cfg, loaders, class_counts, run_dir, device):
    run_dir = Path(run_dir)
    ckpt_path = run_dir / 'best_checkpoint.pth'
    if ckpt_path.exists() or (run_dir / 'history.csv').exists():
        raise FileExistsError('Thư mục run đã có kết quả. Chạy lại cell tạo dữ liệu để mở run mới.')

    seed_everything(cfg['seed'])
    loaders['train'].generator.manual_seed(cfg['seed'])
    model = model.to(device)
    ema = ModelEMA(model, decay=cfg['ema_decay']) if cfg['use_ema'] else None
    criterion = FERLoss(
        class_counts,
        label_smoothing=cfg['label_smoothing'],
        logit_adj_tau=cfg['logit_adj_tau'],
        diversity_weight=cfg['diversity_loss_weight'] if cfg['use_mra'] else 0.0,
    ).to(device)
    optimizer, scheduler = build_optimizer(model, cfg)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg['use_amp'] and device.type == 'cuda')

    total_params = sum(p.numel() for p in model.parameters())
    print(f'Training {cfg["backbone"]} (MRA={cfg["use_mra"]}, img={cfg["image_size"]}) | Params: {total_params:,}')

    best_key = None
    best_epoch = 0
    history = []
    for epoch in range(1, cfg['epochs'] + 1):
        started = perf_counter()
        head_lr = next(g['lr'] for g in optimizer.param_groups if g['name'] == 'head')
        top_bb_lr = max((g['lr'] for g in optimizer.param_groups if g['name'] != 'head'), default=head_lr)

        train_stats = train_one_epoch(model, ema, loaders['train'], optimizer, scaler, criterion, device, cfg, epoch)
        eval_model = ema.ema if ema is not None else model
        val_stats, _, _ = collect_logits(eval_model, loaders['val'], device, cfg, tta_flip=False)

        key = selection_key(val_stats, cfg['selection_metric'])
        improved = best_key is None or key > best_key
        if improved:
            best_key, best_epoch = key, epoch
        stop_early = 0 < cfg['early_stop_patience'] <= (epoch - best_epoch) and epoch < cfg['epochs']

        clean_acc = None
        if epoch % cfg['clean_train_every'] == 0 or epoch == cfg['epochs'] or stop_early:
            clean_stats, _, _ = collect_logits(eval_model, loaders['clean_train'], device, cfg, tta_flip=False)
            clean_acc = clean_stats['accuracy']

        if improved:
            save_checkpoint(eval_model, cfg, epoch, val_stats, class_counts, ckpt_path)
            (run_dir / 'best_validation.json').write_text(
                json.dumps({'epoch': epoch, **val_stats}, indent=2), encoding='utf-8'
            )
            print(f'  ★ Saved best (ep {epoch}): Val Acc {val_stats["accuracy"]:.2%} | Macro F1 {val_stats["macro_f1"]:.2%} | NLL {val_stats["nll"]:.4f}')

        elapsed = perf_counter() - started
        row = {
            'epoch': epoch,
            'train_loss': train_stats['loss'],
            'train_cls_loss': train_stats['cls_loss'],
            'train_div_loss': train_stats['div_loss'],
            'train_accuracy': train_stats['accuracy'],
            'clean_train_accuracy': clean_acc,
            'val_nll': val_stats['nll'],
            'val_ece': val_stats['ece'],
            'val_accuracy': val_stats['accuracy'],
            'val_macro_f1': val_stats['macro_f1'],
            'val_balanced_accuracy': val_stats['balanced_accuracy'],
            'val_composite': val_stats['composite'],
            'lr_backbone_top': top_bb_lr,
            'lr_head': head_lr,
            'epoch_seconds': elapsed,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
        print(
            f'Epoch {epoch:02d}/{cfg["epochs"]} | Tr {train_stats["accuracy"]:.2%} | '
            f'Val Acc {val_stats["accuracy"]:.2%} | F1 {val_stats["macro_f1"]:.2%} | '
            f'NLL {val_stats["nll"]:.4f} | ECE {val_stats["ece"]:.4f} | '
            f'LR {top_bb_lr:.2e}/{head_lr:.2e} | {elapsed:.1f}s'
        )
        if clean_acc is not None:
            print(f'  Clean train acc ({len(loaders["clean_train"].dataset)} ảnh): {clean_acc:.2%} | Gap: {(clean_acc - val_stats["accuracy"]) * 100:.2f} pp')
        if stop_early:
            print(f'Early stop tại epoch {epoch}/{cfg["epochs"]} (best epoch {best_epoch}, patience {cfg["early_stop_patience"]}).')
            break
        scheduler.step()

    best_model, payload = load_trained_model(ckpt_path, device)
    return best_model, payload, pd.DataFrame(history)


## 5. Huấn luyện 1 pha (Single-Phase Cosine + EMA)

Không chạy pha pilot tách rời; toàn bộ ngân sách 35 epoch chạy một lịch Cosine liên tục cùng Model EMA.


In [ ]:
seed_everything(CFG['seed'])
model = build_model(CFG, pretrained=CFG['pretrained'])
total_params = sum(p.numel() for p in model.parameters())
training_metadata = {
    'config': CFG,
    'model_config': model_spec(CFG),
    'classes': list(CLASS_NAMES),
    'data_root': str(DATA_ROOT),
    'data_summary': data_summary,
    'split_fingerprint': SPLIT_FINGERPRINT,
    'class_counts': CLASS_COUNTS,
    'parameter_count': total_params,
    'torch_version': torch.__version__,
    'torchvision_version': torchvision.__version__,
    'checkpoint_file': 'best_checkpoint.pth',
}
(RUN_DIR / 'training.json').write_text(json.dumps(training_metadata, indent=2), encoding='utf-8')

best_model, best_payload, history = run_training(model, CFG, loaders, CLASS_COUNTS, RUN_DIR, DEVICE)
del model
training_metadata['training_summary'] = {
    'planned_epochs': CFG['epochs'],
    'completed_epochs': len(history),
    'best_epoch': best_payload['epoch'],
    'total_train_seconds': float(history['epoch_seconds'].sum()),
}
(RUN_DIR / 'training.json').write_text(json.dumps(training_metadata, indent=2), encoding='utf-8')
print(f'Hoàn tất train trong {history["epoch_seconds"].sum() / 60:.1f} phút | Best epoch: {best_payload["epoch"]}')


## 6. Đánh giá Validation, Calibration (Temperature Scaling), HFlip TTA & Post-hoc Logit Adjustment

Mọi phân tích sau huấn luyện (Temperature Scaling, HFlip TTA, quét `posthoc_tau` trên phân phối train prior)
chỉ thực hiện trên **Validation**; không chạm tới tập `test/`.


In [ ]:
def export_split_report(logits, targets, run_dir, split_name, paths=None):
    run_dir = Path(run_dir)
    metrics = compute_metrics_from_logits(logits, targets)
    true = targets.long().cpu().numpy()
    preds = logits.float().argmax(dim=1).cpu().numpy()
    (run_dir / f'{split_name}_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    report = classification_report(true, preds, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES, digits=4, zero_division=0)
    (run_dir / f'{split_name}_report.txt').write_text(report, encoding='utf-8')
    cm = confusion_matrix(true, preds, labels=list(range(len(CLASS_NAMES))))
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(run_dir / f'{split_name}_confusion.csv')
    pred_df = {'target': true, 'prediction': preds}
    if paths is not None:
        pred_df['path'] = paths
    pd.DataFrame(pred_df).to_csv(run_dir / f'{split_name}_predictions.csv', index=False)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set(title=f'{split_name} confusion matrix', xlabel='Predicted', ylabel='True')
    fig.tight_layout()
    fig.savefig(run_dir / f'{split_name}_confusion.png', dpi=150)
    plt.show()
    print(split_name, json.dumps(metrics, indent=2))
    print(report)
    return metrics


def benchmark_latency_ms(model, image_size, device, batch_sizes=(1, 64), runs=20):
    model.eval()
    results = {}
    with torch.inference_mode():
        for bs in batch_sizes:
            x = torch.randn(bs, 3, image_size, image_size, device=device)
            for _ in range(3):
                _ = model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            started = perf_counter()
            for _ in range(runs):
                _ = model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            elapsed_ms = (perf_counter() - started) * 1000.0 / runs
            results[f'batch_{bs}_ms_per_batch'] = round(elapsed_ms, 2)
            results[f'batch_{bs}_ms_per_image'] = round(elapsed_ms / bs, 3)
    return results


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
history.plot(x='epoch', y=['train_loss', 'val_nll'], ax=axes[0], title='Train loss vs Unsmoothed Val NLL')
history.plot(x='epoch', y=['train_accuracy', 'val_accuracy', 'val_macro_f1'], ax=axes[1], title='Accuracy & Macro F1')
clean_df = history.dropna(subset=['clean_train_accuracy'])
if not clean_df.empty:
    axes[1].plot(clean_df['epoch'], clean_df['clean_train_accuracy'], 'o--', label='clean train')
axes[1].legend()
history.plot(x='epoch', y=['lr_backbone_top', 'lr_head'], ax=axes[2], title='Cosine LR (top backbone / head)')
fig.tight_layout()
fig.savefig(RUN_DIR / 'training_history.png', dpi=150)
plt.show()

val_paths = split_manifest.loc[split_manifest['split'] == 'val', 'path'].tolist()
val_single_metrics, val_single_logits, val_targets = collect_logits(best_model, loaders['val'], DEVICE, CFG, tta_flip=False)
print('=== Validation (Single-View) ===')
export_split_report(val_single_logits, val_targets, RUN_DIR, 'validation_single', val_paths)

if CFG['eval_tta_flip']:
    val_tta_metrics, val_tta_logits, _ = collect_logits(best_model, loaders['val'], DEVICE, CFG, tta_flip=True)
    print(f'=== Validation (HFlip TTA) === Acc {val_tta_metrics["accuracy"]:.2%} | Macro F1 {val_tta_metrics["macro_f1"]:.2%} | NLL {val_tta_metrics["nll"]:.4f}')
    active_val_logits = val_tta_logits
else:
    val_tta_metrics = None
    active_val_logits = val_single_logits

# Calibration (Temperature Scaling) trên validation.
temperature = fit_temperature(active_val_logits, val_targets)
cal_metrics = compute_metrics_from_logits(active_val_logits / temperature, val_targets)
print(f'Temperature Scaling T={temperature:.3f} -> Val NLL {cal_metrics["nll"]:.4f} | Val ECE {cal_metrics["ece"]:.4f}')

# Quét Post-hoc Logit Adjustment (chỉ dùng train priors, không retrain).
priors = torch.tensor(CLASS_COUNTS, dtype=torch.float32)
log_priors = (priors / priors.sum()).log()
pareto_rows = []
for tau in (0.0, 0.2, 0.4, 0.6, 0.8):
    adj_logits = active_val_logits - tau * log_priors
    m = compute_metrics_from_logits(adj_logits, val_targets)
    pareto_rows.append({'posthoc_tau': tau, 'val_accuracy': m['accuracy'], 'val_macro_f1': m['macro_f1'], 'val_balanced_acc': m['balanced_accuracy']})
pareto_df = pd.DataFrame(pareto_rows)
pareto_df.to_csv(RUN_DIR / 'val_posthoc_logit_adj.csv', index=False)
print('Post-hoc Logit Adjustment Pareto trên Validation:')
print(pareto_df.to_string(index=False))

latency = benchmark_latency_ms(best_model, CFG['image_size'], DEVICE, batch_sizes=(1, min(64, CFG['batch_size'])), runs=10)
print('Inference latency:', json.dumps(latency))
posthoc_summary = {
    'use_tta_flip': CFG['eval_tta_flip'],
    'temperature': temperature,
    'calibrated_val': cal_metrics,
    'best_posthoc_tau_by_macro_f1': float(pareto_df.loc[pareto_df['val_macro_f1'].idxmax(), 'posthoc_tau']),
    'latency': latency,
}
save_checkpoint(best_model, CFG, best_payload['epoch'], val_single_metrics, CLASS_COUNTS, RUN_DIR / 'best_checkpoint.pth', extra=posthoc_summary)
(RUN_DIR / 'posthoc_validation.json').write_text(json.dumps(posthoc_summary, indent=2), encoding='utf-8')


## 7. Đánh giá Test cuối cùng (chỉ bật sau khi đã chốt hoàn toàn mô hình bằng Validation)

Giữ `RUN_FINAL_TEST = False` trong toàn bộ quá trình so sánh preset và đa seed.
Khi bật `True`, cell sẽ:
- Nạp lại `best_checkpoint.pth` độc lập;
- Kiểm tra và báo cáo số ảnh trùng pixel tuyệt đối (MD5) giữa `train/` và `test/` để minh bạch chất lượng benchmark;
- Xuất kết quả Single-View và (nếu bật) HFlip TTA + `POSTHOC_TAU` đã chọn sẵn trên validation (không tinh chỉnh trên test).


In [ ]:
def evaluate_final_test(run_dir, root, device, posthoc_tau=0.0):
    run_dir = Path(run_dir)
    metadata = json.loads((run_dir / 'training.json').read_text(encoding='utf-8'))
    cfg = metadata['config']
    restored, payload = load_trained_model(run_dir / metadata['checkpoint_file'], device)
    test_ds = datasets.ImageFolder(Path(root) / 'test', transform=build_transforms(cfg, training=False))
    check_classes(test_ds)
    test_cache = attach_gray_cache(cfg, test_ds)

    # Báo cáo minh bạch tỷ lệ trùng pixel exact MD5 giữa train manifest và test/.
    manifest = pd.read_csv(run_dir / 'split_manifest.csv')
    train_rel_paths = manifest.loc[manifest['split'] == 'train', 'path'].tolist()
    train_md5s = {
        hashlib.md5(decode_gray_array(Path(root) / rel).tobytes()).hexdigest()
        for rel in train_rel_paths
    }
    test_overlap = sum(
        1 for p, _ in test_ds.samples
        if hashlib.md5((test_cache.get_array(p) if test_cache is not None else decode_gray_array(p)).tobytes()).hexdigest() in train_md5s
    )
    print(f'Exact pixel MD5 overlap train vs test: {test_overlap}/{len(test_ds)} ({test_overlap / max(1, len(test_ds)):.2%})')

    num_workers = resolve_num_workers(cfg)
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg['batch_size'],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=device.type == 'cuda',
        worker_init_fn=seed_worker,
    )
    paths = [str(Path(p).relative_to(root)) for p, _ in test_ds.samples]
    _, test_logits, test_targets = collect_logits(restored, test_loader, device, cfg, tta_flip=cfg['eval_tta_flip'])
    if posthoc_tau > 0:
        counts = torch.tensor(payload['class_counts'], dtype=torch.float32)
        log_priors = (counts / counts.sum()).log()
        test_logits = test_logits - posthoc_tau * log_priors
    metrics = export_split_report(test_logits, test_targets, run_dir, 'test', paths)
    print(f'FINAL TEST | Acc {metrics["accuracy"]:.2%} | Macro F1 {metrics["macro_f1"]:.2%} | NLL {metrics["nll"]:.4f} | ECE {metrics["ece"]:.4f}')
    return metrics


RUN_FINAL_TEST = False
POSTHOC_TAU = 0.0  # Chỉ đổi sang giá trị đã chọn trên Validation (val_posthoc_logit_adj.csv).
if RUN_FINAL_TEST:
    test_metrics = evaluate_final_test(RUN_DIR, DATA_ROOT, DEVICE, posthoc_tau=POSTHOC_TAU)
else:
    print('RUN_FINAL_TEST=False — chưa đọc test/. Tiếp tục đối chứng trên Validation.')
